**IMPORT LIBRARIES**

In [1]:
%pip install requests
%pip install beautifulsoup4


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [8]:
import requests 
from bs4 import BeautifulSoup
import csv
import pandas as pd 
import time 

**SEND URL AND SEND GET request**

In [9]:
url=('https://books.toscrape.com/')
response =requests.get(url)
if response.status_code==200:
    print('requst sucessfull')
else:
    print('request failed')


requst sucessfull


**PARSE HTML**

In [10]:
soup=BeautifulSoup(response.text,'html.parser')
print(soup)

<!DOCTYPE html>

<!--[if lt IE 7]>      <html lang="en-us" class="no-js lt-ie9 lt-ie8 lt-ie7"> <![endif]-->
<!--[if IE 7]>         <html lang="en-us" class="no-js lt-ie9 lt-ie8"> <![endif]-->
<!--[if IE 8]>         <html lang="en-us" class="no-js lt-ie9"> <![endif]-->
<!--[if gt IE 8]><!--> <html class="no-js" lang="en-us"> <!--<![endif]-->
<head>
<title>
    All products | Books to Scrape - Sandbox
</title>
<meta content="text/html; charset=utf-8" http-equiv="content-type"/>
<meta content="24th Jun 2016 09:29" name="created"/>
<meta content="" name="description"/>
<meta content="width=device-width" name="viewport"/>
<meta content="NOARCHIVE,NOCACHE" name="robots"/>
<!-- Le HTML5 shim, for IE6-8 support of HTML elements -->
<!--[if lt IE 9]>
        <script src="//html5shim.googlecode.com/svn/trunk/html5.js"></script>
        <![endif]-->
<link href="static/oscar/favicon.ico" rel="shortcut icon"/>
<link href="static/oscar/css/styles.css" rel="stylesheet" type="text/css"/>
<link href="s

**Exteract Book Details From Page One**

In [11]:
books = soup.find_all("h3")

start_time = time.time()
books_data = []

# Loop through each book container
for book in books:
    # Extract relative URL and build absolute link
    relative_url = book.find("a")["href"]
    if not relative_url.startswith("catalogue/"):
        relative_url = "catalogue/" + relative_url
    full_url = url + relative_url

    # Fetch individual book page
    book_response = requests.get(full_url)
    book_soup = BeautifulSoup(book_response.content, "html.parser")

    # Extract required fields
    title = book_soup.find("h1").text
    category = book_soup.find("ul", class_="breadcrumb").find_all("a")[2].text.strip()
    rating = book_soup.find("p", class_="star-rating")["class"][1]
    price = book_soup.find("p", class_="price_color").text.strip()
    availability = book_soup.find("p", class_="availability").text.strip()

    # Store in dictionary container
    books_data.append({
        "title": title,
        "category": category,
        "rating": rating,
        "price": price,
        "availability": availability
    })

end_time = time.time()
total_time = (end_time - start_time) / 60.0

print(f"Successfully extracted {len(books_data)} books in {total_time:.2f} minutes.")
print("\nFirst book record preview:")
print(books_data[0])

Successfully extracted 20 books in 0.69 minutes.

First book record preview:
{'title': 'A Light in the Attic', 'category': 'Poetry', 'rating': 'Three', 'price': '£51.77', 'availability': 'In stock (22 available)'}


In [12]:
import re

raw_price = books_data[0]["price"]
price_number = float(re.search(r"\d+\.\d+", raw_price).group())

print(f"Original: {raw_price}")
print(f"Clean Float: {price_number}")

Original: £51.77
Clean Float: 51.77


In [14]:
raw_stock = books_data[0]["availability"]
stock_number = int(re.search(r"\d+", raw_stock).group())

print(f"Original: {raw_stock}")
print(f"Clean Integer: {stock_number}")

Original: In stock (22 available)
Clean Integer: 22


In [13]:
import re

rating_map = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}
clean_books_data = []

for book in books_data:
    # 1. Clean price to float
    price_num = float(re.search(r"\d+\.\d+", book["price"]).group())

    # 2. Clean stock to integer
    stock_num = int(re.search(r"\d+", book["availability"]).group())

    # 3. Clean rating word to integer number
    rating_num = rating_map.get(book["rating"], 0)

    # 4. Save to clean list
    clean_books_data.append({
        "title": book["title"],
        "category": book["category"],
        "rating": rating_num,
        "price": price_num,
        "availability": stock_num
    })

print(f"Cleaned all {len(clean_books_data)} books!")
print("Cleaned preview of Book 1:")
print(clean_books_data[0])

Cleaned all 20 books!
Cleaned preview of Book 1:
{'title': 'A Light in the Attic', 'category': 'Poetry', 'rating': 3, 'price': 51.77, 'availability': 22}


In [21]:
#[ ] Phase 4: JSON Export

import json

json.dump(clean_books_data, open("books_data.json", "w"), indent=4)
print("Saved books_data.json successfully!")

Saved books_data.json successfully!


with open("books_data.json", "w") as f: = Open a file named "books_data.json" in write mode, and call that open file container f.

json = The target JSON module.

.dump() = The Action (pour Python data into a file).

(clean_books_data, f, indent=4) = The Instructions:

clean_books_data = The data to pour in.

f = The file container to write into.

indent=4 = Format it with clean 4-space indents so it's easy for humans to read.

In [25]:

df= pd.read_json("books_data.json")

df.head()

,title,category,rating,price,availability
0,A Light in the Attic,Poetry,3,51.77,22
1,Tipping the Velvet,Historical Fiction,1,53.74,20
2,Soumission,Fiction,1,50.10,20
3,Sharp Objects,Mystery,4,47.82,20
4,Sapiens: A Brief History of Humankind,History,5,54.23,20


In [30]:
# Top 5 most expensive books
df.sort_values(by="price", ascending=False)[["title", "category", "price"]].head(5)


,title,category,price
15,Our Band Could Be Your Life: Scenes from the A...,Music,57.25
4,Sapiens: A Brief History of Humankind,History,54.23
1,Tipping the Velvet,Historical Fiction,53.74
13,Scott Pilgrim's Precious Little Life (Scott Pi...,Sequential Art,52.29
9,The Black Maria,Poetry,52.15


In [32]:
df.groupby('category')['price'].mean()

category
Business              33.340000
Default               18.173333
Fiction               50.100000
Historical Fiction    53.740000
History               54.230000
Music                 46.135000
Mystery               47.820000
Poetry                37.115000
Politics              51.330000
Science Fiction       37.590000
Sequential Art        52.290000
Travel                45.170000
Young Adult           20.055000
Name: price, dtype: float64

In [33]:
df[df['rating']==5][['title','category','price']]

,title,category,price
4,Sapiens: A Brief History of Humankind,History,54.23
12,Set Me Free,Young Adult,17.46
13,Scott Pilgrim's Precious Little Life (Scott Pi...,Sequential Art,52.29
14,Rip it Up and Start Again,Music,35.02


In [34]:
df.to_csv("books_data.csv", index=False)